In [ ]:
df = 0 

In [ ]:
def iqr_filter(group):

    q1 = group['month_spend'].quantile(0.25)
    q3 = group['month_spend'].quantile(0.75)

    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return group[
        group['month_spend'].between(lower, upper)
    ]


clean_df = (
    df
    .groupby('campaigns_cnt', group_keys=False)
    .apply(iqr_filter)
    .reset_index(drop=True)
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


# -----------------------------
# Форматирование оси Y
# -----------------------------

def format_y_axis():

    plt.gca().yaxis.set_major_formatter(
        mticker.FuncFormatter(
            lambda x, _: f'{x:,.0f}'.replace(',', ' ')
        )
    )


# =========================================================
# 1. РТО когорт по календарным месяцам
# =========================================================

calendar_stats = (
    clean_df
    .groupby(
        ['campaigns_cnt', 'month_dt'],
        as_index=False
    )
    .agg(
        avg_spend_per_client=('month_spend', 'mean'),
        median_spend_per_client=('month_spend', 'median')
    )
)

calendar_stats = calendar_stats.sort_values(
    ['campaigns_cnt', 'month_dt']
)


# -----------------------------
# Медианный РТО - календарные месяцы
# -----------------------------

plt.figure(figsize=(14, 7))

for cohort in sorted(calendar_stats['campaigns_cnt'].unique()):

    part = calendar_stats[
        calendar_stats['campaigns_cnt'] == cohort
    ]

    plt.plot(
        part['month_dt'],
        part['median_spend_per_client'],
        marker='o',
        linewidth=2,
        label=f'{cohort} камп.'
    )

plt.title('Медианный РТО когорт по календарным месяцам')

plt.xlabel('Месяц')
plt.ylabel('Медианный РТО')

plt.grid(True, alpha=0.3)

plt.legend(
    title='Кол-во кампаний',
    bbox_to_anchor=(1.02, 1),
    loc='upper left'
)

format_y_axis()

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


# =========================================================
# 2. Поведение относительно первой кампании
# =========================================================

shift_stats = (
    clean_df
    .groupby(
        ['campaigns_cnt', 'month_shift'],
        as_index=False
    )
    .agg(
        avg_spend_per_client=('month_spend', 'mean'),
        median_spend_per_client=('month_spend', 'median')
    )
)

shift_stats = shift_stats.sort_values(
    ['campaigns_cnt', 'month_shift']
)


# -----------------------------
# Медианный РТО - относительно первой кампании
# -----------------------------

plt.figure(figsize=(14, 7))

for cohort in sorted(shift_stats['campaigns_cnt'].unique()):

    part = shift_stats[
        shift_stats['campaigns_cnt'] == cohort
    ]

    plt.plot(
        part['month_shift'],
        part['median_spend_per_client'],
        marker='o',
        linewidth=2,
        label=f'{cohort} камп.'
    )

plt.axvline(
    x=0,
    linestyle='--',
    alpha=0.5
)

plt.xticks(
    [-1, 0, 1, 2, 3, 4],
    [
        '-1 мес',
        '1-я камп.',
        '+1',
        '+2',
        '+3',
        '+4'
    ]
)

plt.title(
    'Медианный РТО относительно первой кампании'
)

plt.xlabel('Период относительно первой кампании')
plt.ylabel('Медианный РТО')

plt.grid(True, alpha=0.3)

plt.legend(
    title='Кол-во кампаний',
    bbox_to_anchor=(1.02, 1),
    loc='upper left'
)

format_y_axis()

plt.tight_layout()
plt.show()